# 01 Data Understanding
## DataCo Supply Chain : Late Delivery Risk Prediction

**Goal of this notebook:**
- Load the raw dataset and take a first look
- Understand what each column represents
- Group the 53 columns into logical domains
- Identify: useful columns / useless columns / leakage risk
- Document decisions that will guide the cleaning phase
---

In [ ]:
# ===========================================
# Imports
# ===========================================
import pandas as pd
import numpy as np

## Load Dataset
First look at the raw data - shape, size, and first rows.

In [5]:
# ===========================================
# Load Dataset
# ===========================================
df = pd.read_csv('../data/DataCoSupplyChainDataset.csv',
                 encoding='latin-1')
print(f' Shape: {df.shape[0]} rows x {df.shape[1]} columns')

 Shape: 180519 rows x 53 columns


## First Look
it is a Quick preview of the data - first rows, column names, and the data types

In [6]:
# ===========================================
# First Look at the Data
# ===========================================

# First 5 rows
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [10]:
# ===========================================
# All Column Names
# ===========================================

for i, col in enumerate(df.columns, 1):
    print(f'{i}. {col}')

1. Type
2. Days for shipping (real)
3. Days for shipment (scheduled)
4. Benefit per order
5. Sales per customer
6. Delivery Status
7. Late_delivery_risk
8. Category Id
9. Category Name
10. Customer City
11. Customer Country
12. Customer Email
13. Customer Fname
14. Customer Id
15. Customer Lname
16. Customer Password
17. Customer Segment
18. Customer State
19. Customer Street
20. Customer Zipcode
21. Department Id
22. Department Name
23. Latitude
24. Longitude
25. Market
26. Order City
27. Order Country
28. Order Customer Id
29. order date (DateOrders)
30. Order Id
31. Order Item Cardprod Id
32. Order Item Discount
33. Order Item Discount Rate
34. Order Item Id
35. Order Item Product Price
36. Order Item Profit Ratio
37. Order Item Quantity
38. Sales
39. Order Item Total
40. Order Profit Per Order
41. Order Region
42. Order State
43. Order Status
44. Order Zipcode
45. Product Card Id
46. Product Category Id
47. Product Description
48. Product Image
49. Product Name
50. Product Price
51

## Data Types & Missing Values
Check the type of each column and how many null values exist.

In [12]:
# ===========================================
# Data Types
# ===========================================
print(df.dtypes.to_string())

Type                              object
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Benefit per order                float64
Sales per customer               float64
Delivery Status                   object
Late_delivery_risk                 int64
Category Id                        int64
Category Name                     object
Customer City                     object
Customer Country                  object
Customer Email                    object
Customer Fname                    object
Customer Id                        int64
Customer Lname                    object
Customer Password                 object
Customer Segment                  object
Customer State                    object
Customer Street                   object
Customer Zipcode                 float64
Department Id                      int64
Department Name                   object
Latitude                         float64
Longitude                        float64
Market          

### Cleaning Note
order date (DateOrders) -> object | !! datetime

shipping date (DateOrders) -> object | !! datetime

In [14]:
# ===========================================
# Missing Values
# ===========================================
df.isnull().sum()

Type                                  0
Days for shipping (real)              0
Days for shipment (scheduled)         0
Benefit per order                     0
Sales per customer                    0
Delivery Status                       0
Late_delivery_risk                    0
Category Id                           0
Category Name                         0
Customer City                         0
Customer Country                      0
Customer Email                        0
Customer Fname                        0
Customer Id                           0
Customer Lname                        8
Customer Password                     0
Customer Segment                      0
Customer State                        0
Customer Street                       0
Customer Zipcode                      3
Department Id                         0
Department Name                       0
Latitude                              0
Longitude                             0
Market                                0


### Note for data cleaning phase (delete this)

Customer Lname -> 8 nulls | PII = Personally Identifiable Information


Customer Zipcode -> 3 nulls | PII = Personally Identifiable Information


Order Zipcode -> 155,679 nulls (!!!) | Most of it is null, Not useful for ML models 

Product Description -> 180,519 nulls (100% Nulls) (!!!)


---
## Domain Mapping
Group the 53 columns into logical domains to better understand the dataset

In [16]:
# ===========================================
# Domain Mapping (From Data Description) 
# 53 columns into 6 groups
# ===========================================

customer_cols = [
    'Customer Id', 'Customer Fname', 'Customer Lname',
    'Customer Email', 'Customer Password', 'Customer Segment',
    'Customer City', 'Customer Country', 'Customer State',
    'Customer Street', 'Customer Zipcode'
]

product_cols = [
    'Product Card Id', 'Product Category Id', 'Category Id',
    'Category Name', 'Product Name', 'Product Description',
    'Product Image', 'Product Price', 'Product Status'
]

order_cols = [
    'Order Id', 'Order Customer Id', 'order date (DateOrders)',
    'Order Status', 'Order Item Id', 'Order Item Cardprod Id',
    'Order Item Quantity', 'Type'
]

finance_cols = [
    'Sales', 'Order Item Total', 'Order Item Product Price',
    'Order Item Discount', 'Order Item Discount Rate',
    'Benefit per order', 'Order Item Profit Ratio',
    'Order Profit Per Order', 'Sales per customer'
]

shipping_cols = [
    'Shipping Mode', 'shipping date (DateOrders)',
    'Days for shipping (real)', 'Days for shipment (scheduled)',
    'Delivery Status', 'Late_delivery_risk'
]

geo_cols = [
    'Market', 'Order Region', 'Order Country', 'Order City',
    'Order State', 'Order Zipcode', 'Latitude', 'Longitude',
    'Department Id', 'Department Name'
]


all_grouped = (customer_cols + product_cols + order_cols +
               finance_cols + shipping_cols + geo_cols)

print(f"Customer  columns : {len(customer_cols)}")
print(f"Product   columns : {len(product_cols)}")
print(f"Order     columns : {len(order_cols)}")
print(f"Finance   columns : {len(finance_cols)}")
print(f"Shipping  columns : {len(shipping_cols)}")
print(f"Geo       columns : {len(geo_cols)}")
print(f"{'─'*30}")
print(f"Total             : {len(all_grouped)}")
print(f"Match 53?         : {len(all_grouped) == 53} ")

Customer  columns : 11
Product   columns : 9
Order     columns : 8
Finance   columns : 9
Shipping  columns : 6
Geo       columns : 10
──────────────────────────────
Total             : 53
Match 53?         : True 


---

## Group Analysis
Examine each domain group: Unique values, sample data, and initial decisions

In [17]:
# ===========================================
# Customer Group
# ===========================================
print('Customer Columns:')

for col in customer_cols:
    unique = df[col].nunique()
    sample = df[col].iloc[0]

    print(f'{col} | Unique: {unique} | Sample: {sample}')

Customer Columns:
Customer Id | Unique: 20652 | Sample: 20755
Customer Fname | Unique: 782 | Sample: Cally
Customer Lname | Unique: 1109 | Sample: Holloway
Customer Email | Unique: 1 | Sample: XXXXXXXXX
Customer Password | Unique: 1 | Sample: XXXXXXXXX
Customer Segment | Unique: 3 | Sample: Consumer
Customer City | Unique: 563 | Sample: Caguas
Customer Country | Unique: 2 | Sample: Puerto Rico
Customer State | Unique: 46 | Sample: PR
Customer Street | Unique: 7458 | Sample: 5365 Noble Nectar Island
Customer Zipcode | Unique: 995 | Sample: 725.0


### Customer Group: Decision (keep this)
* customer Segment | Keep | 3 categories, useful feature |
* Customer Country | Keep | 2 unique values, may be useful |

---

In [18]:
# ===========================================
# Product Group
# ===========================================

for col in product_cols:
    unique = df[col].nunique()
    sample = df[col].iloc[0]

    print(f'{col} | Unique: {unique} | Sample: {sample}')

Product Card Id | Unique: 118 | Sample: 1360
Product Category Id | Unique: 51 | Sample: 73
Category Id | Unique: 51 | Sample: 73
Category Name | Unique: 50 | Sample: Sporting Goods
Product Name | Unique: 118 | Sample: Smart watch 
Product Description | Unique: 0 | Sample: nan
Product Image | Unique: 118 | Sample: http://images.acmesports.sports/Smart+watch 
Product Price | Unique: 75 | Sample: 327.75
Product Status | Unique: 1 | Sample: 0


### Notes

1. Product Category Id = Category Id | identical columns

2. Product Status has only one unique value across all the rows, This meaning "it adds (NO INFORMATION)" to the model

### Product Group: Decision

* Keep: Category Name, Product Price 
* Drop: All remaining columns

---

In [19]:
# ===========================================
# ORDER GROUP
# ===========================================

for col in order_cols:
    unique = df[col].nunique()
    sample = df[col].iloc[0]

    print(f'{col} | Unique: {unique} | Sample: {sample}')

Order Id | Unique: 65752 | Sample: 77202
Order Customer Id | Unique: 20652 | Sample: 20755
order date (DateOrders) | Unique: 65752 | Sample: 1/31/2018 22:56
Order Status | Unique: 9 | Sample: COMPLETE
Order Item Id | Unique: 180519 | Sample: 180517
Order Item Cardprod Id | Unique: 118 | Sample: 1360
Order Item Quantity | Unique: 5 | Sample: 1
Type | Unique: 4 | Sample: DEBIT


### Order Group — Decision Summary

* Keep: Order Status, Order Item Quantity, Type, order date (as engineered features)
* Reference only: Order Id (kept for traceability, not a model feature)
* Drop: Order Customer Id, Order Item Id, Order Item Cardprod Id

 **Note:**
 
  ID columns (Order Id, Order Customer Id, Order Item Id) are
 sequential identifiers — the model cannot learn anything from them.

Order Id is kept as a reference only, not as a model feature.

order date will be converted from object to datetime and engineered into: month, day of week, and quarter.

---

In [20]:
# ===========================================
# Finance Group
# ===========================================
for col in finance_cols:
    unique = df[col].nunique()
    sample = df[col].iloc[0]

    print(f'{col} | Unique: {unique} | Sample: {sample}')

Sales | Unique: 193 | Sample: 327.75
Order Item Total | Unique: 2927 | Sample: 314.6400146
Order Item Product Price | Unique: 75 | Sample: 327.75
Order Item Discount | Unique: 1017 | Sample: 13.10999966
Order Item Discount Rate | Unique: 18 | Sample: 0.039999999
Benefit per order | Unique: 21998 | Sample: 91.25
Order Item Profit Ratio | Unique: 162 | Sample: 0.289999992
Order Profit Per Order | Unique: 21998 | Sample: 91.25
Sales per customer | Unique: 2927 | Sample: 314.6400146


### Finanace Group: Decision
* Keep: Order Item Product Price, Order Item Discount Rate,
         Order Item Profit Ratio, Benefit per order
* Drop:

1.  Sales (derived from Price × Quantity × (1 - Discount Rate))
2.  Order Item Total = Sales per customer (identical columns)
3.  Order Profit Per Order = Benefit per order (identical columns)
4. Order Item Discount (Discount Rate is more useful — normalized)

---


In [21]:
# ===========================================
# Shipping Group
# ===========================================
for col in shipping_cols:
    unique = df[col].nunique()
    sample = df[col].iloc[0]

    print(f'{col} | Unique: {unique} | Sample: {sample}')

Shipping Mode | Unique: 4 | Sample: Standard Class
shipping date (DateOrders) | Unique: 63701 | Sample: 2/3/2018 22:56
Days for shipping (real) | Unique: 7 | Sample: 3
Days for shipment (scheduled) | Unique: 4 | Sample: 4
Delivery Status | Unique: 4 | Sample: Advance shipping
Late_delivery_risk | Unique: 2 | Sample: 0


### Shipping Group — Decision Summary

* Keep: Shipping Mode, Days for shipment (scheduled), shipping date (as engineered features)
* Target: Late_delivery_risk
* Drop: Days for shipping (real), Delivery Status — DATA LEAKAGE

 **Note — Data Leakage Explained:**
 When a new order arrives, we only know: Shipping Mode, Scheduled Days, Market, and order details. We do NOT yet know how many days shipping
 actually took or whether it was late — these values only exist after delivery happens. Giving these columns to the model means giving it the answer before the exam — it would not learn to predict, it would just read the result.

 **Important:** Before dropping Days for shipping (real), we will use it to engineer delay_gap = Days for shipping (real) − Days for shipment (scheduled)
 This will be the target for Track 2 (Regression).

In [22]:
# ===========================================
# GEO Group
# ===========================================
for col in geo_cols:
    unique = df[col].nunique()
    sample = df[col].iloc[0]

    print(f'{col} | Unique: {unique} | Sample: {sample}')

Market | Unique: 5 | Sample: Pacific Asia
Order Region | Unique: 23 | Sample: Southeast Asia
Order Country | Unique: 164 | Sample: Indonesia
Order City | Unique: 3597 | Sample: Bekasi
Order State | Unique: 1089 | Sample: Java Occidental
Order Zipcode | Unique: 609 | Sample: nan
Latitude | Unique: 11250 | Sample: 18.2514534
Longitude | Unique: 4487 | Sample: -66.03705597
Department Id | Unique: 11 | Sample: 2
Department Name | Unique: 11 | Sample: Fitness


### Geo Group — Decision Summary

*  Keep: Market, Order Region, Latitude, Longitude, Department Name

* Drop
1. Order Country — Market already captures geography at a higher level (5 markets vs 164 countries)
2.  Order City, Order State — too granular, too many unique values (3,597 and 1,089)
3.  Order Zipcode — too granular + contains null values
4. Department Id — Department Name is more readable, same information

 **Note:** Latitude and Longitude are kept for Track 3 and Track 4,
 where geographic clustering and route pattern discovery require precise location data.

---

---
##  Final Summary — All 53 Columns

### Columns to Keep 

**Customer:** Customer Segment, Customer Country  

**Product:** Category Name, Product Price  

**Order:** Order Id (reference), Order Status, Order Item Quantity, Type, order date  

**Finance:** Order Item Product Price, Order Item Discount Rate, Benefit per order, Order Item Profit Ratio 

**Shipping:** Shipping Mode, Days for shipment (scheduled), shipping date  

**Geo:** Market, Order Region, Latitude, Longitude, Department Name 

**Target:** Late_delivery_risk 

**Engineered:** delay_gap, order_month, order_dayofweek, order_quarter  

### Columns to Drop 

**PII:** Customer Fname, Customer Lname, Customer Email, Customer Password, Customer Street  

**Too Granular:** Customer City, Customer State, Customer Zipcode, Order City, Order State, Order Zipcode, Order Country  

**Leakage:** Days for shipping (real), Delivery Status  

**Duplicates:** Product Category Id, Order Item Cardprod Id, Department Id, Order Profit Per Order, Sales per customer, Order Item Total  

**Useless:** Product Description, Product Image, Product Name, Product Status, Product Card Id, Sales, Order Item Discount, Order Customer Id, Order Item Id

---

## Correlation with Target
Quick look at how each numeric column relates to **Late_delivery_risk.**

This will guide our feature selection in the modeling phase.

In [23]:
# ===========================================
# CORRELATION WITH TARGET
# ===========================================

# Select numeric columns only
numeric_cols = df.select_dtypes(include='number').columns.tolist()

# Remove leakage and target from correlation
exclude = ['Late_delivery_risk', 
           'Days for shipping (real)']

numeric_cols = [col for col in numeric_cols if col not in exclude]

# Calculate correlation with target
corr = df[numeric_cols].corrwith(df['Late_delivery_risk'])
corr = corr.sort_values(ascending=False)

print("Correlation with Late_delivery_risk:")
print("=" * 50)
for col, val in corr.items():
    print(f"{col:<40} | {val:.4f}")

Correlation with Late_delivery_risk:
Customer Zipcode                         | 0.0031
Product Category Id                      | 0.0018
Category Id                              | 0.0018
Order Item Cardprod Id                   | 0.0015
Product Card Id                          | 0.0015
Customer Id                              | 0.0015
Order Customer Id                        | 0.0015
Department Id                            | 0.0011
Latitude                                 | 0.0007
Order Item Discount Rate                 | 0.0004
Order Item Quantity                      | -0.0001
Order Item Discount                      | -0.0007
Order Id                                 | -0.0013
Order Item Id                            | -0.0014
Longitude                                | -0.0019
Product Price                            | -0.0022
Order Item Product Price                 | -0.0022
Order Item Profit Ratio                  | -0.0023
Sales                                    | -0.0036
Orde

c:\Users\suras\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\suras\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


### Correlation Summary

**Days for shipment (scheduled)** is the only numeric column with a meaningful correlation (-0.37) with the target.
 All other numeric columns show near-zero correlation.

 This suggests that categorical features (Market, Shipping Mode, Customer Segment) will be more important predictors to be explored in 03_eda.ipynb.

### Why Categorical Features Matter More

Correlation measures the relationship between numeric columns and the target.
From the results, almost all numeric columns scored near zero — meaning
price, quantity, and discount have little direct relationship with late delivery.

This does not mean our data is weak. It means the real story is hiding
in the categorical columns. A model does not care about the order price
but it does care whether the shipment went through LATAM via Standard Class.

We cannot measure this with correlation alone. Instead, in `03_eda.ipynb`
we will group by each categorical column and measure the late delivery rate
directly — that is where the patterns will appear.